In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)



In [17]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [18]:
def run_prompt(test_case):
    """Runs the prompt with the given test case input"""
    prompt = test_case["task"]
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [19]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [23]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [24]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

    results = run_eval(dataset)

In [26]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS CloudFormation Template Parser\n\nHere's a comprehensive solution with error handling and examples:\n\n```python\nimport json\nfrom typing import List, Optional\n\ndef parse_cloudformation_template(template_json: str) -> List[str]:\n    \"\"\"\n    Parse an AWS CloudFormation template (JSON string) and return a list \n    of all resource logical IDs.\n    \n    Args:\n        template_json (str): CloudFormation template as a JSON string\n        \n    Returns:\n        List[str]: List of resource logical IDs\n        \n    Raises:\n        json.JSONDecodeError: If the template is not valid JSON\n        KeyError: If the template is missing the 'Resources' section\n        TypeError: If template_json is not a string\n    \"\"\"\n    if not isinstance(template_json, str):\n        raise TypeError(f\"Expected string, got {type(template_json).__name__}\")\n    \n    try:\n        template = json.loads(template_json)\n    except json.JSONDecodeError as e:\n       